In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils import preprocess_data, get_features_and_target
from regression_utils import (
    train_and_compare_regression,
    print_results_table
)

In [3]:
#Подготовка данных
df = preprocess_data('data.xlsx')
X, y = get_features_and_target(df, 'CC50, mM', task_type='regression')
print(f"\nЦелевая переменная: CC50 (логарифмированная через log1p)")
print(f"Размер X: {X.shape}, размер y: {y.shape}")
print(f"\nСтатистика y (после log1p):")
print(f"  min:    {y.min():.3f}")
print(f"  max:    {y.max():.3f}")
print(f"  mean:   {y.mean():.3f}")
print(f"  median: {y.median():.3f}")
print(f"  std:    {y.std():.3f}")

Загружено: 1001 объектов, 213 столбцов
Удалено константных признаков: 33
Пропуски заполнены медианой каждого столбца
Удалено признаков с корреляцией > 0.95: 33
Итого: 1001 объектов, 147 столбцов (144 признаков)

Целевая переменная: CC50 (логарифмированная через log1p)
Размер X: (1001, 144), размер y: (1001,)

Статистика y (после log1p):
  min:    0.531
  max:    8.421
  mean:   5.567
  median: 6.021
  std:    1.587


In [4]:
# Запускаем обучение всех моделей с подбором гиперпараметров
results_df, best_models, splits = train_and_compare_regression(
    X, y,
    target_name='CC50',
    cv=5,
    test_size=0.3,
    random_state=42
)

X_train, X_test, y_train, y_test = splits

In [ ]:
#Вывод итоговой таблицы
print_results_table(results_df, target_name='CC50')

from pathlib import Path; Path('results').mkdir(parents=True, exist_ok=True)
results_df.to_csv('results/results_regression_CC50.csv', index=False)


Итоговая таблица сравнений для CC50
           Model     CV R2  Train R2  Test R2  Test MAE  Test RMSE
             KNN  0.393193  0.870126 0.405946  0.816268   1.165204
    RandomForest  0.442069  0.819363 0.376309  0.865598   1.193915
             SVR  0.417260  0.781905 0.345892  0.859741   1.222682
GradientBoosting  0.419138  0.854374 0.299822  0.893472   1.265007
           Lasso  0.254927  0.300888 0.286743  1.061707   1.276768
           Ridge -0.119458  0.548206 0.221399  0.998222   1.333971
LinearRegression -0.147869  0.549596 0.189518  1.014984   1.361008
    DecisionTree  0.177160  0.499145 0.000720  1.170867   1.511237

 Лучшая модель: KNN (Test R2 = 0.4059) ***

Таблица результатов сохранена: results/results_regression_CC50.csv


In [6]:
# Сравнение моделей
plt.figure(figsize=(12, 6))
sorted_df = results_df.sort_values('Test R2', ascending=True)
plt.barh(sorted_df['Model'], sorted_df['Test R2'], color='steelblue', edgecolor='black')
plt.xlabel('R^2 на тестовой выборке')
plt.title('Сравнение моделей регрессии по R^2 (CC50)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig('results/comparison_CC50.png', dpi=100, bbox_inches='tight')
plt.close()


# Предсказания vs реальные
best_model_name = results_df.iloc[0]['Model']
best_model = best_models[best_model_name]
y_pred_test = best_model.predict(X_test)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred_test, alpha=0.5, color='steelblue')
min_val = min(y_test.min(), y_pred_test.min())
max_val = max(y_test.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Идеальное предсказание')
plt.xlabel('Истинное log(1 + CC50)')
plt.ylabel('Предсказанное log(1 + CC50)')
plt.title(f'Предсказания vs реальные значения\n(лучшая модель: {best_model_name})')
plt.legend()
plt.tight_layout()
plt.savefig('results/predictions_CC50.png', dpi=100, bbox_inches='tight')
plt.close()

In [8]:
# Важность признаков
final_model = best_model.named_steps['model']
if hasattr(final_model, 'feature_importances_'):
    print(f"\n--- Топ-15 важных признаков для модели {best_model_name} ---")
    importances = pd.Series(final_model.feature_importances_, index=X.columns)
    top_features = importances.sort_values(ascending=False).head(15)
    for feat, imp in top_features.items():
        print(f"  {feat:30s}: {imp:.4f}")

    plt.figure(figsize=(10, 6))
    top_features.sort_values().plot(kind='barh', color='coral', edgecolor='black')
    plt.xlabel('Важность признака')
    plt.title(f'Топ-15 признаков для предсказания CC50\n({best_model_name})')
    plt.tight_layout()
    plt.savefig('results/feature_importance_CC50.png', dpi=100, bbox_inches='tight')
    plt.close()